# Run all benchmarks

Runs every benchmark notebook in this folder against **one model** and saves each notebook's executed report (with its results dashboard) into a folder named after the model.

Set the model, URL and key in the configuration cell. They are handed to every notebook and **override `.env`** (and the prompts), so you never need to edit the individual notebooks. Proxy settings (`PROXY_URL`, `NO_PROXY`, `CA_BUNDLE`) are still read from `.env` if you have one.

For each benchmark you get, in `<RESULTS_ROOT>/<model name>/`:

- `run-<name>.ipynb` — the notebook as executed, with all output and charts
- `run-<name>.html` — the same report as a standalone page
- `run-<name>.log` — the runner's log for that notebook
- `summary.json` and `.eval-notebook-scorecard.json` — pass/fail and scores for the whole run

**Requires:** `nbconvert` and `ipykernel` in this notebook's Python environment (JupyterLab already has them; otherwise `pip install nbconvert ipykernel`), plus everything the individual notebooks need (`uv`, Node.js for promptfoo). Full benchmarks can take hours; set `LIMIT = 10` in each notebook first to smoke-test.

In [ ]:
# --- Model under test (overrides .env for every notebook) ---
MODEL_URL = "https://your-model-endpoint.example.com/v1"   # OpenAI-compatible base URL, usually ends in /v1
MODEL_NAME = "your-model-name"                              # exactly as the endpoint knows it; also the report folder name
MODEL_API_KEY = ""                                          # leave empty if the endpoint needs no key

# --- Output ---
RESULTS_ROOT = "."   # reports go to RESULTS_ROOT/<model name>/ (relative to this notebook's folder)

# --- Which notebooks to run, in order. Comment a line out to skip it. ---
NOTEBOOKS = [
    "run-ifeval.ipynb",
    "run-math.ipynb",
    "run-mmlu.ipynb",
    "run-truthfulqa.ipynb",
    "run-reasoning.ipynb",
    "run-multilingual.ipynb",
    "run-toolcalling.ipynb",
    "run-promptfoo.ipynb",
]
RUN_SCORECARD = True          # finish with run-scorecard.ipynb, a one-page summary of this run
KEEP_GOING = True             # if a notebook errors, carry on with the rest (False = stop)
TIMEOUT_PER_NOTEBOOK = None   # seconds allowed per notebook; None = no limit

In [ ]:
import json, os, re, shutil, subprocess, sys, time
from pathlib import Path

# The notebooks (and eval_env.py) sit in this folder; the kernel may have started in the parent folder.
NB_DIR = next(d for d in (Path("."), Path("notebooks")) if (d / "eval_env.py").is_file()).resolve()

assert MODEL_NAME.strip() and "your-model-name" not in MODEL_NAME, "Set MODEL_NAME in the configuration cell"
assert MODEL_URL.strip() and "example.com" not in MODEL_URL, "Set MODEL_URL in the configuration cell"

# One folder per model; "/" and other awkward characters in the name become "_"
REPORT_DIR = (NB_DIR / RESULTS_ROOT / re.sub(r"[^\w.\-]+", "_", MODEL_NAME.strip()).strip("._") ).resolve()
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Handed to eval_env.setup() in every notebook; wins over .env. Only the child processes see these.
CHILD_ENV = {**os.environ,
             "EVAL_OVERRIDE_URL": MODEL_URL.strip(), "EVAL_OVERRIDE_MODEL": MODEL_NAME.strip(),
             "EVAL_OVERRIDE_API_KEY": MODEL_API_KEY}

# Use this environment's nbconvert if it has one, otherwise fetch one with uvx.
try:
    import nbconvert, ipykernel  # noqa: F401
    NBCONVERT = [sys.executable, "-m", "nbconvert"]
except ImportError:
    NBCONVERT = ["uvx", "--with", "ipykernel", "--with", "matplotlib", "--from", "nbconvert", "jupyter-nbconvert"]

def execute(nb_name):
    """Run one notebook top to bottom in NB_DIR and save the executed copy + HTML into REPORT_DIR.
    Returns (ok, seconds, detail)."""
    stem = Path(nb_name).stem
    executed = REPORT_DIR / f"{stem}.ipynb"
    start = time.time()
    # --allow-errors keeps going past a failing cell so a partial report is still saved; errors are counted below.
    cmd = NBCONVERT + ["--to", "notebook", "--execute", "--allow-errors",
                       f"--ExecutePreprocessor.timeout={-1 if TIMEOUT_PER_NOTEBOOK is None else TIMEOUT_PER_NOTEBOOK}",
                       "--output-dir", str(REPORT_DIR), "--output", stem, str(NB_DIR / nb_name)]
    with open(REPORT_DIR / f"{stem}.log", "w") as log:
        rc = subprocess.run(cmd, cwd=NB_DIR, env=CHILD_ENV, stdout=log, stderr=subprocess.STDOUT).returncode
        if rc == 0:
            subprocess.run(NBCONVERT + ["--to", "html", "--output-dir", str(REPORT_DIR), "--output", stem, str(executed)],
                           cwd=NB_DIR, stdout=log, stderr=subprocess.STDOUT)
    secs = time.time() - start
    if rc != 0 or not executed.is_file():
        return False, secs, f"nbconvert exited with {rc} (see {stem}.log)"
    nb = json.loads(executed.read_text())
    errors = [o for c in nb["cells"] if c["cell_type"] == "code" for o in c.get("outputs", []) if o["output_type"] == "error"]
    if errors:
        return False, secs, f"{errors[0]['ename']}: {errors[0]['evalue']}"[:300]
    # Pick up the PASS / FAIL line the notebook printed
    text = "".join(o.get("text", "") if isinstance(o.get("text", ""), str) else "".join(o["text"])
                   for c in nb["cells"] if c["cell_type"] == "code" for o in c.get("outputs", []) if o["output_type"] == "stream")
    verdict = next((l for l in reversed(text.splitlines()) if l.startswith(("PASS", "FAIL"))), "finished")
    return True, secs, verdict

# Start the scorecard afresh so it only reflects this model
(NB_DIR / ".eval-notebook-scorecard.json").unlink(missing_ok=True)

print(f"Model: {MODEL_NAME}  @ {MODEL_URL}\nReports: {REPORT_DIR}\n")
summary = {"model": MODEL_NAME, "url": MODEL_URL, "started": time.strftime("%Y-%m-%d %H:%M:%S"), "notebooks": {}}
todo = list(NOTEBOOKS)
for i, nb_name in enumerate(todo, 1):
    print(f"[{i}/{len(todo)}] {nb_name} ...", flush=True)
    ok, secs, detail = execute(nb_name)
    summary["notebooks"][nb_name] = {"ran_ok": ok, "seconds": round(secs), "result": detail}
    print(f"    {'done' if ok else 'ERROR'} in {secs/60:.1f} min - {detail}", flush=True)
    if not ok and not KEEP_GOING:
        break

if RUN_SCORECARD:
    print("[scorecard] run-scorecard.ipynb ...", flush=True)
    ok, secs, detail = execute("run-scorecard.ipynb")
    summary["notebooks"]["run-scorecard.ipynb"] = {"ran_ok": ok, "seconds": round(secs), "result": detail}
    print(f"    {'done' if ok else 'ERROR'} - {detail}")
    if (NB_DIR / ".eval-notebook-scorecard.json").is_file():
        shutil.copy(NB_DIR / ".eval-notebook-scorecard.json", REPORT_DIR / ".eval-notebook-scorecard.json")

summary["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
(REPORT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))

In [ ]:
print(f"Results for {MODEL_NAME} - reports in {REPORT_DIR}\n")
for name, r in summary["notebooks"].items():
    print(f"{'ok   ' if r['ran_ok'] else 'ERROR'}  {name:<26} {r['seconds']:>6}s  {r['result']}")